<a href="https://colab.research.google.com/github/Madhaveffai/pytorch-learning/blob/main/week2/Overfitting_%26_Regularization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Deliberately Overfit

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

X = torch.linspace(-3, 3, 80).unsqueeze(1)
y = torch.sin(X) + 0.3 * torch.randn(80, 1)

dataset = TensorDataset(X, y)
train_set, val_set = random_split(dataset, [60, 20])
train_loader = DataLoader(train_set, batch_size = 16, shuffle = True)
val_loader = DataLoader(val_set, batch_size=20, shuffle=False)

class BigMLP(nn.Module):
  def __init__(self, use_dropout=False):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(1, 128), nn.ReLU(),
        nn.Dropout(0.3) if use_dropout else nn.Identity(),
        nn.Linear(128,128), nn.ReLU(),
        nn.Dropout(0.3) if use_dropout else nn.Identity(),
        nn.Linear(128, 1)
    )
  def forward(self, x):
    return self.net(x)

Training Function

In [23]:
def train(model, epochs=200, weight_decay=0.0):
  optimizer = torch.optim.Adam(
      model.parameters(), lr=0.01, weight_decay=weight_decay
  )
  loss_fn = nn.MSELoss()

  for epoch in range(epochs):
    model.train()
    for X_b, y_b in train_loader:
      loss = loss_fn(model(X_b), y_b)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    if epoch % 50 == 0:
      model.eval()
      with torch.no_grad():
        val_loss = 0
        train_loss = 0
        for X_b, y_b in val_loader:
            val_loss += loss_fn(model(X_b), y_b).item()
        for X_b, y_b in train_loader:
            train_loss += loss_fn(model(X_b), y_b).item()
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
      print(f"Epoch {epoch} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

  print()

Compare all three

In [24]:
print("=== No regularization ===")
train(BigMLP())

print("=== with Dropout ===")
train(BigMLP(use_dropout=True))

print("=== with weight decay ===")
train(BigMLP(), weight_decay=1e-3)

=== No regularization ===
Epoch 0 | Train: 0.7052 | Val: 0.5109
Epoch 50 | Train: 0.0528 | Val: 0.0802
Epoch 100 | Train: 0.0579 | Val: 0.0669
Epoch 150 | Train: 0.0535 | Val: 0.0807

=== with Dropout ===
Epoch 0 | Train: 0.2756 | Val: 0.2088
Epoch 50 | Train: 0.0765 | Val: 0.0753
Epoch 100 | Train: 0.0553 | Val: 0.0756
Epoch 150 | Train: 0.0557 | Val: 0.0856

=== with weight decay ===
Epoch 0 | Train: 0.3209 | Val: 0.2785
Epoch 50 | Train: 0.0680 | Val: 0.0845
Epoch 100 | Train: 0.0604 | Val: 0.0777
Epoch 150 | Train: 0.0546 | Val: 0.0804

